<h3 style="color: #e0e0ff; font-style: italic;">🧬 Multimodal Variational Autoencoder (MVAE) for Multimodal Fake News Detection</h3>

---

**MVAE Concept & Architecture**

The **Multimodal Variational Autoencoder (MVAE)** learns a shared, robust latent representation of text and image modalities. It detects misinformation by capturing cross-modal correlations and discrepancies in a low-dimensional latent space.

**Mathematical Formulation**

**1. Modal Encoding**: Text $x_t \in \mathbb{R}^{768}$ and image features $x_v \in \mathbb{R}^{2048}$ are projected and concatenated: $x = [x_t, x_v]$

**2. Latent Mapping**: $\mu = W_\mu h + b_\mu, \quad \log(\sigma^2) = W_\sigma h + b_\sigma$

**3. Reparameterization Trick**: $z = \mu + \epsilon \odot \exp\left(\frac{1}{2}\log(\sigma^2)\right), \quad \epsilon \sim \mathcal{N}(0, I)$

**4. Reconstruction**: Decoders reconstruct $\hat{x}_t$ and $\hat{x}_v$ from $z$.

**5. Loss Function** (ELBO + Classification):
$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{cls}} + \alpha \mathcal{L}_{\text{recon}} + \beta \mathcal{L}_{\text{KL}}$$

> **Implementation note**: We use a **Joint MVAE** where both modalities are concatenated before encoding — simpler and faster than Product-of-Experts when both modalities are always available.


**Imports and Setup**

In [1]:
import os
import sys
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score, roc_auc_score, confusion_matrix,
    precision_recall_fscore_support, roc_curve, f1_score
)
from sklearn.manifold import TSNE

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import models
from transformers import XLMRobertaModel, XLMRobertaTokenizer

warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.abspath('../../src'))
from data_pipeline import M4FCDataPipeline
from model_registry import update_registry


c:\Users\SAFAE ERAJI\Desktop\M2\Stage PFE\multimodal_fake_news_detection\venv_ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🚀 Device: {device}')

def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

os.makedirs('../../models/mvae', exist_ok=True)
os.makedirs('../../reports/mvae/figures', exist_ok=True)
os.makedirs('../../reports/mvae/metrics', exist_ok=True)


🚀 Device: cuda


**Dataset & Data Loader**

In [3]:
class MVAEDataset(Dataset):
    """MVAE-specific dataset: raw images for ResNet50 + XLM-RoBERTa text + metadata."""
    def __init__(self, df, tokenizer, max_text_length=128, image_size=(224, 224)):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_text_length = max_text_length
        self.image_size = image_size
        self.text_field = 'multilingual_claim'

    def __len__(self):
        return len(self.df)

    def load_image(self, image_path):
        try:
            if os.path.exists(image_path):
                image = Image.open(image_path).convert('RGB').resize(self.image_size)
                image = torch.from_numpy(np.array(image)).float().permute(2, 0, 1) / 255.0
                mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
                std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
                return (image - mean) / std
        except Exception:
            pass
        return torch.zeros(3, self.image_size[0], self.image_size[1])

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row[self.text_field]) if pd.notna(row[self.text_field]) else ""
        encoding = self.tokenizer(
            text, max_length=self.max_text_length,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        image = self.load_image(row['full_image_path'] if pd.notna(row['full_image_path']) else "")

        is_ai      = float(row['is_ai_generated'])    if pd.notna(row['is_ai_generated'])    else 0.0
        img_exists = float(row['image_exists'])        if pd.notna(row['image_exists'])        else 0.0
        word_count = float(row['claim_word_count'])    if pd.notna(row['claim_word_count'])    else 0.0
        word_count = word_count / 50.0
        manipulated = float(row['is_manipulated_fake']) if pd.notna(row['is_manipulated_fake']) else 0.0
        use_caption = float(row['use_true_caption'])   if pd.notna(row['use_true_caption'])   else 0.0
        metadata = torch.tensor([is_ai, img_exists, word_count, manipulated, use_caption], dtype=torch.float)

        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'image':    image,
            'metadata': metadata,
            'label':    torch.tensor(row['target'], dtype=torch.long)
        }


In [4]:
tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')
pipeline = M4FCDataPipeline(csv_path='../../data/M4FC.csv', target_col='target', random_state=42)
df = pipeline._load_data()

train_df, val_df, test_df = pipeline.split_dataframe(df)
print(f'✓ Dataset split: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}')

BATCH_SIZE = 16
train_dataset = MVAEDataset(train_df, tokenizer)
val_dataset   = MVAEDataset(val_df,   tokenizer)
test_dataset  = MVAEDataset(test_df,  tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)


✓ Dataset split: Train=3462, Val=743, Test=743


**MVAE Model Definition**

In [5]:
class MVAE(nn.Module):
    def __init__(self, text_encoder, image_encoder, latent_dim=128, num_classes=2):
        super().__init__()
        self.text_encoder  = text_encoder
        self.image_encoder = image_encoder

        self.metadata_encoder = nn.Sequential(
            nn.Linear(5, 32), nn.ReLU(), nn.Linear(32, 64)
        )

        # Projection to shared dimension
        self.text_proj  = nn.Linear(768,  512)
        self.image_proj = nn.Linear(2048, 512)

        # VAE Encoder
        self.enc_shared = nn.Sequential(
            nn.Linear(1024, 512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 256),  nn.ReLU()
        )
        self.fc_mu     = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)

        # VAE Decoders
        self.dec_shared = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 512),        nn.ReLU()
        )
        self.dec_text  = nn.Linear(512, 768)
        self.dec_image = nn.Linear(512, 2048)

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim + 64, 128),
            nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def forward(self, input_ids, attention_mask, images, metadata):
        with torch.no_grad():
            text_feats  = self.text_encoder(input_ids, attention_mask=attention_mask).pooler_output
            image_feats = self.image_encoder(images)

        t_proj = F.relu(self.text_proj(text_feats))
        i_proj = F.relu(self.image_proj(image_feats))

        combined = torch.cat([t_proj, i_proj], dim=-1)
        h = self.enc_shared(combined)
        mu, logvar = self.fc_mu(h), self.fc_logvar(h)
        z = self.reparameterize(mu, logvar)

        dec_h = self.dec_shared(z)
        recon_text  = self.dec_text(dec_h)
        recon_image = self.dec_image(dec_h)

        meta_feats = self.metadata_encoder(metadata)
        logits = self.classifier(torch.cat([z, meta_feats], dim=-1))

        return logits, recon_text, recon_image, mu, logvar, text_feats, image_feats


**Training Pipeline**

In [6]:
class MVAETrainer:
    def __init__(self, model, train_loader, val_loader, test_loader,
                 device='cuda', alpha=0.1, beta=0.01, patience=10):
        self.model        = model.to(device)
        self.train_loader = train_loader
        self.val_loader   = val_loader
        self.test_loader  = test_loader
        self.device       = device
        self.alpha        = alpha
        self.beta         = beta
        self.patience     = patience

        self.optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=20)

        self.best_model_path = '../../models/mvae/best_mvae_model.pth'
        self.best_val_f1     = 0.0

        self.metrics_history = {
            'train_loss': [], 'val_loss': [],
            'train_class_loss': [], 'train_recon_loss': [], 'train_kl_loss': [],
            'train_acc': [], 'val_acc': [],
            'precision': [], 'recall': [], 'f1': [], 'auc': []
        }

    def calculate_loss(self, logits, labels, recon_text, recon_image, orig_text, orig_image, mu, logvar):
        """ELBO loss = CrossEntropy + alpha * Reconstruction MSE + beta * KL Divergence."""
        class_loss = F.cross_entropy(logits, labels)
        recon_loss = F.mse_loss(recon_text, orig_text) + F.mse_loss(recon_image, orig_image)
        kl_loss    = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / logits.size(0)
        total_loss = class_loss + self.alpha * recon_loss + self.beta * kl_loss
        return total_loss, class_loss, recon_loss, kl_loss

    def train_epoch(self):
        self.model.train()
        total_loss = total_class = total_recon = total_kl = 0.0
        predictions, labels_list = [], []

        for batch in tqdm(self.train_loader, desc='Training'):
            input_ids      = batch['input_ids'].to(self.device)
            attention_mask = batch['attention_mask'].to(self.device)
            images         = batch['image'].to(self.device)
            metadata       = batch['metadata'].to(self.device)
            labels         = batch['label'].to(self.device)

            self.optimizer.zero_grad()
            logits, recon_text, recon_image, mu, logvar, text_feats, image_feats = \
                self.model(input_ids, attention_mask, images, metadata)

            loss, c_loss, r_loss, k_loss = self.calculate_loss(
                logits, labels, recon_text, recon_image, text_feats, image_feats, mu, logvar
            )
            loss.backward()
            nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.optimizer.step()

            total_loss  += loss.item()
            total_class += c_loss.item()
            total_recon += r_loss.item()
            total_kl    += k_loss.item()
            predictions.extend(torch.argmax(logits, dim=1).cpu().numpy())
            labels_list.extend(labels.cpu().numpy())

        self.scheduler.step()
        n = len(self.train_loader)
        return (total_loss/n, total_class/n, total_recon/n, total_kl/n,
                accuracy_score(labels_list, predictions),
                f1_score(labels_list, predictions, average='binary'))

    @torch.no_grad()
    def evaluate(self, data_loader, mode='val'):
        self.model.eval()
        total_loss = 0.0
        predictions, probabilities, labels_list = [], [], []

        for batch in tqdm(data_loader, desc=f'Evaluating {mode}'):
            input_ids      = batch['input_ids'].to(self.device)
            attention_mask = batch['attention_mask'].to(self.device)
            images         = batch['image'].to(self.device)
            metadata       = batch['metadata'].to(self.device)
            labels         = batch['label'].to(self.device)

            logits, recon_text, recon_image, mu, logvar, text_feats, image_feats = \
                self.model(input_ids, attention_mask, images, metadata)

            loss, _, _, _ = self.calculate_loss(
                logits, labels, recon_text, recon_image, text_feats, image_feats, mu, logvar
            )
            total_loss += loss.item()

            probs = F.softmax(logits, dim=1)
            predictions.extend(torch.argmax(logits, dim=1).cpu().numpy())
            probabilities.extend(probs[:, 1].cpu().numpy())
            labels_list.extend(labels.cpu().numpy())

        precision, recall, f1, _ = precision_recall_fscore_support(
            labels_list, predictions, average='binary', zero_division=0
        )
        return {
            'loss':        total_loss / len(data_loader),
            'accuracy':    accuracy_score(labels_list, predictions),
            'precision':   precision,
            'recall':      recall,
            'f1':          f1,
            'auc':         roc_auc_score(labels_list, probabilities),
            'predictions': predictions,
            'true_labels': labels_list,
            'probabilities': probabilities
        }

    def train(self, epochs=100):
        print('Starting MVAE Training Pipeline...')
        patience_counter = 0

        for epoch in range(epochs):
            print(f'\nEpoch {epoch+1}/{epochs}')
            train_loss, train_class, train_recon, train_kl, train_acc, train_f1 = self.train_epoch()
            val_metrics = self.evaluate(self.val_loader, 'val')

            self.metrics_history['train_loss'].append(train_loss)
            self.metrics_history['train_class_loss'].append(train_class)
            self.metrics_history['train_recon_loss'].append(train_recon)
            self.metrics_history['train_kl_loss'].append(train_kl)
            self.metrics_history['train_acc'].append(train_acc)
            self.metrics_history['val_loss'].append(val_metrics['loss'])
            self.metrics_history['val_acc'].append(val_metrics['accuracy'])
            self.metrics_history['precision'].append(val_metrics['precision'])
            self.metrics_history['recall'].append(val_metrics['recall'])
            self.metrics_history['f1'].append(val_metrics['f1'])
            self.metrics_history['auc'].append(val_metrics['auc'])

            print(f'Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f}')
            print(f'Val   Loss: {val_metrics["loss"]:.4f} | Acc: {val_metrics["accuracy"]:.4f} | F1: {val_metrics["f1"]:.4f} | AUC: {val_metrics["auc"]:.4f}')

            if val_metrics['f1'] > self.best_val_f1:
                self.best_val_f1 = val_metrics['f1']
                patience_counter = 0
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': self.model.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'val_f1': val_metrics['f1']
                }, self.best_model_path)
                print(f'  Best model saved! Val F1={val_metrics["f1"]:.4f}')
            else:
                patience_counter += 1
                if patience_counter >= self.patience:
                    print(f'Early stopping at epoch {epoch+1}')
                    break

        return self.metrics_history

    def test(self):
        checkpoint = torch.load(self.best_model_path, weights_only=False)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        start = time.time()
        test_metrics = self.evaluate(self.test_loader, 'test')
        test_metrics['inference_time'] = time.time() - start
        print(f'Accuracy: {test_metrics["accuracy"]:.4f} | F1: {test_metrics["f1"]:.4f} | AUC: {test_metrics["auc"]:.4f}')
        return test_metrics


**Model Initialization & Training**

In [7]:
print('Loading pre-trained encoders (XLM-RoBERTa + ResNet50)...')
text_encoder  = XLMRobertaModel.from_pretrained('xlm-roberta-base')
image_encoder = models.resnet50(weights='IMAGENET1K_V1')
image_encoder.fc = nn.Identity()

for param in text_encoder.parameters():  param.requires_grad = False
for param in image_encoder.parameters(): param.requires_grad = False

model   = MVAE(text_encoder, image_encoder, latent_dim=128, num_classes=2).to(device)
trainer = MVAETrainer(model, train_loader, val_loader, test_loader, device=device, patience=10)

history = trainer.train(epochs=100)
test_metrics = trainer.test()


Loading pre-trained encoders (XLM-RoBERTa + ResNet50)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2926.36it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Starting MVAE Training Pipeline...

Epoch 1/100


Evaluating val: 100%|██████████| 47/47 [00:33<00:00,  1.41it/s]


Train Loss: 0.4127 | Acc: 0.8758
Val   Loss: 0.2702 | Acc: 0.9408 | F1: 0.9695 | AUC: 0.5260
  Best model saved! Val F1=0.9695

Epoch 2/100


...

Train Loss: 0.1580 | Acc: 0.9596
Val   Loss: 0.2410 | Acc: 0.9354 | F1: 0.9663 | AUC: 0.8057

Epoch 16/100


Evaluating val: 100%|██████████| 47/47 [00:40<00:00,  1.15it/s]


Train Loss: 0.1533 | Acc: 0.9596
Val   Loss: 0.2334 | Acc: 0.9341 | F1: 0.9657 | AUC: 0.8078
Early stopping at epoch 16


Evaluating test: 100%|██████████| 47/47 [00:43<00:00,  1.08it/s]

Accuracy: 0.9394 | F1: 0.9687 | AUC: 0.8676


**Evaluation & Visualization**

In [8]:
class MVAEVisualizer:
    FIG_DIR = '../../reports/mvae/figures'

    @classmethod
    def _save(cls, name):
        os.makedirs(cls.FIG_DIR, exist_ok=True)
        path = os.path.join(cls.FIG_DIR, name)
        plt.tight_layout()
        plt.savefig(path, dpi=100, bbox_inches='tight')
        plt.close()
        print(f'Saved: {path}')

    @classmethod
    def plot_training_history(cls, history):
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        axes[0,0].plot(history['train_loss'], label='Train Loss')
        axes[0,0].plot(history['val_loss'],   label='Val Loss')
        axes[0,0].set_title('Loss'); axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

        axes[0,1].plot(history['train_acc'], label='Train Acc')
        axes[0,1].plot(history['val_acc'],   label='Val Acc')
        axes[0,1].set_title('Accuracy'); axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)

        axes[1,0].plot(history['f1'],  label='F1',  marker='o')
        axes[1,0].plot(history['auc'], label='AUC', marker='s')
        axes[1,0].set_title('F1 & AUC'); axes[1,0].legend(); axes[1,0].grid(True, alpha=0.3)

        axes[1,1].plot(history['precision'], label='Precision', marker='^')
        axes[1,1].plot(history['recall'],    label='Recall',    marker='v')
        axes[1,1].set_title('Precision-Recall'); axes[1,1].legend(); axes[1,1].grid(True, alpha=0.3)

        cls._save('training_history.png')

    @classmethod
    def plot_loss_components(cls, history):
        plt.figure(figsize=(8, 5))
        plt.plot(history['train_class_loss'], label='Classification Loss', color='crimson')
        plt.plot(history['train_recon_loss'], label='Reconstruction Loss', color='teal')
        plt.plot(history['train_kl_loss'],    label='KL Divergence',       color='orange')
        plt.title('MVAE Loss Components'); plt.xlabel('Epoch'); plt.ylabel('Loss')
        plt.legend(); plt.grid(True, alpha=0.3)
        cls._save('loss_components.png')

    @classmethod
    def plot_confusion_matrix(cls, y_true, y_pred):
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=['Real','Fake'], yticklabels=['Real','Fake'])
        plt.title('MVAE Confusion Matrix'); plt.ylabel('Actual'); plt.xlabel('Predicted')
        cls._save('confusion_matrix.png')

    @classmethod
    def plot_roc_curve(cls, y_true, y_prob):
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        auc_val = roc_auc_score(y_true, y_prob)
        plt.figure(figsize=(7, 6))
        plt.plot(fpr, tpr, color='darkorange', lw=2.5, label=f'AUC = {auc_val:.3f}')
        plt.plot([0,1],[0,1], '--', color='navy')
        plt.fill_between(fpr, tpr, alpha=0.2, color='darkorange')
        plt.title('MVAE ROC Curve'); plt.xlabel('FPR'); plt.ylabel('TPR')
        plt.legend(); plt.grid(True, alpha=0.3)
        cls._save('roc_curve.png')

    @classmethod
    def plot_reconstruction_error(cls, real_errors, fake_errors):
        plt.figure(figsize=(8, 5))
        plt.hist(real_errors, bins=30, alpha=0.7, label='Real News', color='teal',    edgecolor='black')
        plt.hist(fake_errors, bins=30, alpha=0.7, label='Fake News', color='crimson', edgecolor='black')
        plt.axvline(np.mean(real_errors), color='teal',    linestyle='--', lw=2, label=f'Real Mean: {np.mean(real_errors):.3f}')
        plt.axvline(np.mean(fake_errors), color='crimson', linestyle='--', lw=2, label=f'Fake Mean: {np.mean(fake_errors):.3f}')
        plt.title('Reconstruction Error Distribution by Class')
        plt.xlabel('Reconstruction Error'); plt.ylabel('Frequency')
        plt.legend(); plt.grid(True, alpha=0.3)
        cls._save('recon_error_by_class.png')

    @classmethod
    def plot_latent_tsne(cls, latent_codes, labels):
        print('Running t-SNE on latent space...')
        latent_2d = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(np.array(latent_codes))
        plt.figure(figsize=(8, 6))
        for val, color, name in [(0, 'teal', 'Real'), (1, 'crimson', 'Fake')]:
            mask = np.array(labels) == val
            plt.scatter(latent_2d[mask,0], latent_2d[mask,1], c=color, label=name, alpha=0.6, s=30)
        plt.title('MVAE Latent Space (t-SNE)')
        plt.xlabel('Dim 1'); plt.ylabel('Dim 2'); plt.legend()
        cls._save('latent_tsne.png')


In [9]:
# Plot standard 4-panel + MVAE-specific plots
MVAEVisualizer.plot_training_history(history)
MVAEVisualizer.plot_loss_components(history)
MVAEVisualizer.plot_confusion_matrix(test_metrics['true_labels'], test_metrics['predictions'])
MVAEVisualizer.plot_roc_curve(test_metrics['true_labels'], test_metrics['probabilities'])

# MVAE-specific: reconstruction error distribution & latent space t-SNE
model.eval()
latent_codes = []
real_errors, fake_errors = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        images         = batch['image'].to(device)
        metadata       = batch['metadata'].to(device)
        labels         = batch['label'].to(device)

        logits, recon_text, recon_image, mu, _, text_feats, image_feats = \
            model(input_ids, attention_mask, images, metadata)

        latent_codes.extend(mu.cpu().numpy())

        t_err = F.mse_loss(recon_text,  text_feats,  reduction='none').mean(dim=1)
        i_err = F.mse_loss(recon_image, image_feats, reduction='none').mean(dim=1)
        for err, lbl in zip((t_err + i_err).cpu().numpy(), labels.cpu().numpy()):
            (real_errors if lbl == 0 else fake_errors).append(err)

MVAEVisualizer.plot_reconstruction_error(real_errors, fake_errors)
MVAEVisualizer.plot_latent_tsne(latent_codes, test_metrics['true_labels'])


Saved: ../../reports/mvae/figures\training_history.png
Saved: ../../reports/mvae/figures\loss_components.png
Saved: ../../reports/mvae/figures\confusion_matrix.png
Saved: ../../reports/mvae/figures\roc_curve.png
Saved: ../../reports/mvae/figures\recon_error_by_class.png
Running t-SNE on latent space...
Saved: ../../reports/mvae/figures\latent_tsne.png


**Reporting & Registry Update**

In [10]:
report_path = '../../reports/mvae/metrics/MVAE_Standard_Report.txt'
os.makedirs(os.path.dirname(report_path), exist_ok=True)

best_val_f1  = max(history['f1'])      if history['f1']      else test_metrics['f1']
best_val_acc = max(history['val_acc']) if history['val_acc'] else test_metrics['accuracy']

with open(report_path, 'w', encoding='utf-8') as f:
    f.write('=' * 50 + '\n')
    f.write('MVAE MULTIMODAL FAKE NEWS DETECTION REPORT\n')
    f.write('=' * 50 + '\n\n')
    f.write(f'Epochs completed:  {len(history["train_loss"])}\n')
    f.write(f'Best Val F1:       {best_val_f1:.6f}\n')
    f.write(f'Best Val Accuracy: {best_val_acc:.6f}\n\n')
    f.write('TEST SET RESULTS\n' + '-' * 30 + '\n')
    f.write(f'  Accuracy:  {test_metrics["accuracy"]:.6f}\n')
    f.write(f'  Precision: {test_metrics["precision"]:.6f}\n')
    f.write(f'  Recall:    {test_metrics["recall"]:.6f}\n')
    f.write(f'  F1-Score:  {test_metrics["f1"]:.6f}\n')
    f.write(f'  AUC:       {test_metrics["auc"]:.6f}\n')

print(f'Report saved to {report_path}')

update_registry(
    name='MVAE',
    arch='MVAE',
    rel_path='models/mvae/best_mvae_model.pth',
    val_accuracy=float(best_val_acc),
    val_f1=float(best_val_f1),
    test_accuracy=float(test_metrics['accuracy']),
    test_f1=float(test_metrics['f1']),
    test_auc=float(test_metrics['auc']),
    latent_dim=128,
    num_classes=2
)


Report saved to ../../reports/mvae/metrics/MVAE_Standard_Report.txt
ℹ️  Registry unchanged — current champion 'Fusion - Metadata Fusion (MLP)' (acc=0.9462) leads 'MVAE' (acc=0.9421).


False